In [ ]:
## UNCOMMENT IF TWO PROCESSES ON PARALLEL ### 
# import os 
# os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = str(.33) # pour 2 process en parallele

In [1]:
path_config = './config_QGSW.py'

In [2]:
import sys
sys.path.append('../../MASSH/mapping')

In [3]:
from src import exp
config = exp.Exp(path_config)

name_experiment: config_QGSW
saveoutputs: True
name_exp_save: config_QGSW
path_save: /bettik/bellemva/MASSH_outputs/final_experiment_hawaii_itg/config_QGSW
tmp_DA_path: /silenus/PROJECTS/pr-data-ocean/bellemva/scratch/final_experiment_hawaii_itg/config_QGSW
flag_plot: 0
init_date: 2012-05-01 00:00:00
final_date: 2012-07-31 23:00:00
assimilation_time_step: 1:00:00
saveoutput_time_step: 1:00:00
plot_time_step: 1 day, 0:00:00
time_obs_min: None
time_obs_max: None
write_obs: True
compute_obs: False
path_obs: /silenus/PROJECTS/pr-data-ocean/bellemva/obs/final_experiment_hawaii_itg/config_QGSW
path_bathymetry: /bettik/bellemva/sad/Bathymetry_hawai.nc
name_var_bathy: {'lon': 'lon', 'lat': 'lat', 'var': 'elevation'}
smooth_wavelength: 36000
path_tidal_velocity: /bettik/bellemva/FES_tide
coriolis_force: True
n_workers: 10
name_lon: lon
name_lat: lat
name_time: time

NAME_DIAG is not set in the configuration file


In [4]:
from src import state as state
State = state.State(config)

super: GRID_GEO
lon_min: 190.0
lon_max: 200
lat_min: 20.0
lat_max: 30.0
dlon: 0.0625
dlat: 0.0625
name_init_mask: None
name_var_mask: {'lon': 'longitude', 'lat': 'latitude', 'var': 'ssh_it1'}
interp_method_mask: nearest



In [ ]:
from src import inv as inv
res = inv.Inv(config,State)

In [6]:
from src import mod as mod
Model = mod.Model(config,State)

super: MOD_QG1L_JAX
name_class: Qgm
name_var: {'SSH': 'ssh_bm'}
name_init_var: {}
dir_model: None
var_to_save: None
upwind: 3
advect_pv: True
advect_tracer: False
dtmodel: 300
time_scheme: Euler
c0: 2.7
filec_aux: None
name_var_c: {'lon': '', 'lat': '', 'var': ''}
cmin: None
cmax: None
solver: spectral
init_from_bc: True
dist_sponge_bc: None
Kdiffus: None
Kdiffus_trac: None
bc_trac: OBC
forcing_tracer_from_bc: False
constant_c: True
constant_f: True
f0: None
tile_size: 32
tile_overlap: 16
path_mdt: None
name_var_mdt: None


super: MOD_SW1L_JAX
name_var: {'U': 'u_it', 'V': 'v_it', 'SSH': 'ssh_it'}
name_init_var: None
name_params: ['He', 'He_offset', 'itg', 'hbcx', 'hbcy']
dir_model: None
var_to_save: ['SSH']
dtmodel: 300
time_scheme: rk4
bc_kind: 1d
bc_island: dirichlet
w_waves: [0.00014051890262680206, 0.0001454441043328608, 0.000137879707490692]
w_names: ['m2', 's2', 'n2']
He_init: 0.95
He_data: None
Ntheta: 2
g: 9.81




In [ ]:
from src import bc as bc
Bc = bc.Bc(config)

In [ ]:
from src import obs as obs
dict_obs = obs.Obs(config,State)

In [ ]:
from src import obsop as obsop
Obsop = obsop.Obsop(config,State,dict_obs,Model)

In [7]:
from src import basis as basis
Basis = basis.Basis(config, State)

super: BASIS_BMaux_JAX
name_mod_var: ssh_bm
flux: False
facns: 1.0
facnlt: 2.0
npsp: 3.5
facpsp: 1.5
file_aux: /home/bellemva/MASSH/mapping/aux/aux_reduced_basis_BM.nc
lmin: 80
lmax: 900.0
factdec: 7.5
tdecmin: 2.0
tdecmax: 20.0
facQ: 1
file_depth: None
name_var_depth: {'lon': '', 'lat': '', 'var': ''}
depth1: 0.0
depth2: 30.0
path_background: None
var_background: None
norm_time: True

super: BASIS_GAUSS3D_JAX
name_mod_var: He
flux: True
time_dependant: True
facns: 3.0
facnlt: 3.0
sigma_D: 970
sigma_T: 25
sigma_Q: 0.015
fcor: 1
normalize_fact: False
time_spinup: None
flag_variable_Q: False
path_sad: None
name_var_sad: {'lon': '', 'lat': '', 'var': ''}

super: BASIS_OFFSET
name_mod_var: He_offset
sigma_B: 0.05

super: BASIS_HBC
name_params: ['hbcx', 'hbcy']
facns: 3.5
facnlt: 2.5
time_dependant: True
sigma_B_bc: 0.02
D_bc: 300
T_bc: 20
Nwaves: 3
Ntheta: 2

super: BASIS_GAUSS_ITG
name_mod_var: itg
facns: 3.5
facnlt: 1.0
D_itg: 300
sigma_Q: 0.000175
Nwaves: 3



In [8]:
import numpy as np
from datetime import datetime,timedelta
nstep_check = int(timedelta(hours=24).total_seconds()//Model.dt)
time_basis = np.arange(0,Model.T[-1]+nstep_check*Model.dt,nstep_check*Model.dt)/24/3600 # Time (in days) for which the basis components will be compute (at each timestep_checkpoint)
# Xb, Q = Basis.set_basis(time_basis,return_q=True) # Q is the standard deviation. To get the variance, use Q^2
Xb, Q = Basis.set_basis(time_basis, return_q=True)

Setting Basis BMaux...
Computing Q
lambda=6.8E+02 nlocs=1.6E+01 tdec=1.6E+01 Q=1.1E-03
lambda=4.8E+02 nlocs=1.7E+01 tdec=1.5E+01 Q=1.5E-03
lambda=3.3E+02 nlocs=2.4E+01 tdec=1.1E+01 Q=1.7E-03
lambda=2.3E+02 nlocs=3.3E+01 tdec=5.4E+00 Q=1.5E-03
lambda=1.6E+02 nlocs=4.5E+01 tdec=4.0E+00 Q=1.2E-03
lambda=1.1E+02 nlocs=6.4E+01 tdec=2.0E+00 Q=7.4E-04
lambda=8.0E+01 nlocs=1.1E+02 tdec=2.0E+00 Q=2.9E-04
Computing Spatial components
Computing Time components
reduced order: 2410653 --> 397476
 reduced factor: 6
lambda=9.7E+02 nlocs=6.1E+01 tdec=2.5E+01 ntime=1.5E+01 Q=5.0E-03
reduced order: 2410653 --> 915
 reduced factor: 2634
nbcx: 14850
nbcy: 15300
reduced order: 1796760 --> 30150
reduced factor: 59


In [ ]:
3*2*5*161*4

19320

In [22]:
Basis.Basis[-2].shape_params_phys

{'hbcS': [3, 2, 5, 161],
 'hbcN': [3, 2, 5, 161],
 'hbcE': [3, 2, 5, 161],
 'hbcW': [3, 2, 5, 161]}

In [24]:
Basis.Basis[-2].n_params_phys

{'hbcS': 4830, 'hbcN': 4830, 'hbcE': 4830, 'hbcW': 4830}

In [9]:
Basis.slice_basis

[slice(0, 397476, None),
 slice(397476, 398391, None),
 slice(398391, 398392, None),
 slice(398392, 428542, None),
 slice(428542, 431854, None)]

In [ ]:
from src import inv as inv
inv.Inv(config,State,Model,dict_obs=dict_obs,Obsop=Obsop,Basis=Basis,Bc=Bc)

In [ ]:
from src import inv as inv
inv.Inv(config,State,Model,dict_obs=dict_obs,Obsop=Obsop,Basis=Basis,Bc=Bc)

## Debug test for QGSW reconstruction 

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import glob
list_X = glob.glob("/silenus/PROJECTS/pr-data-ocean/bellemva/scratch/final_experiment_hawaii_itg/config_QGSW/X_it-2025-03-25_063144.nc")
list_X.sort()

In [ ]:
ds = xr.open_dataset(list_X[-1])

In [ ]:
X = ds.res.values[0]

In [ ]:
from src.tools_4Dvar import Cov 
B = Cov(Q)
Xa = Xb + B.sqr(X)

In [ ]:
time_basis.shape

In [ ]:
Basis.operg(t=time_basis[45],X=Xa,State=State)

In [ ]:
plt.pcolormesh(State.params["He"])#+Model.Models[1].Heb+State.params["He_offset"])
plt.colorbar()

In [ ]:
Basis.slice_basis

In [ ]:
Basis.slice_basis[3]

In [ ]:
X[Basis.slice_basis[3]].shape

In [ ]:
Basis.Basis[3].slice_params

In [ ]:
plt.plot(X[Basis.slice_basis[3]][Basis.Basis[3].slice_params["hbcW"]])

## Minimization trajectory 

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
minimization_trajectory = xr.open_dataset("/silenus/PROJECTS/pr-data-ocean/bellemva/scratch/final_experiment_hawaii_itg/config_QGSW/minimization_trajectory.nc")

In [ ]:
plt.figure(figsize=(15,8))
plt.plot(minimization_trajectory.Nj,minimization_trajectory.J)
plt.yscale("log")   
# plt.xlim(250,350)

In [ ]:
traj_j = minimization_trajectory.J
f_diff = (traj_j[0:-1] - traj_j[1:])/np.maximum(np.abs(traj_j[0:-1]),np.abs(traj_j[1:]))

In [ ]:
f_diff[f_diff<0]=np.nan

In [ ]:
fig, ax1 = plt.subplots(figsize=(15,8))

color = 'tab:red'
ax1.set_xlabel('Call to cost and grad function')
ax1.set_ylabel('ftol', color=color)
ax1.plot(minimization_trajectory.Nj[0:-1], f_diff, color=color)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_yscale("log")
# ax1.set_xlim(325,350)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('gtol', color=color)  # we already handled the x-label with ax1
ax2.plot(minimization_trajectory.Ng[0:-1], minimization_trajectory.G[0:-1]/1.3e4, color=color,alpha=0.6)
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_yscale("log")

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.show()